In [ ]:
# reacher_expert_traj

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/'Colab Notebooks'

/content/drive/MyDrive/Colab Notebooks


In [1]:
%pip install -U pip setuptools wheel
%pip install "imitation>=1.0.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 57.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
             ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 85, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 388, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 99, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendo

In [2]:
import sys
!{sys.executable} -m pip install -U gymnasium gymnasium[mujoco] stable-baselines3[extra]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 135.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [stable-baselines3]


In [3]:
import gymnasium as gym
import mujoco
import sys
import numpy as np
import stable_baselines3 as sb3

print("python:", sys.executable)
print("gymnasium:", gym.__version__, gym.__file__)
print("mujoco:", mujoco.__version__, mujoco.__file__)
print("numpy", np.__version__)
print("sb3", sb3.__version__)

python: /usr/bin/python3
gymnasium: 1.2.3 /usr/local/lib/python3.12/dist-packages/gymnasium/__init__.py
mujoco: 3.4.0 /usr/local/lib/python3.12/dist-packages/mujoco/__init__.py
numpy 2.0.2
sb3 2.7.1


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [4]:
import os
import gymnasium as gym
import torch
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.utils import set_random_seed

ENV_ID = "Reacher-v5"  # v4 vs v5.
N_ENVS = 8             # 4–8 is the sweet spot on most machines
SEED = 0

def make_env(rank: int, seed: int = 0):
    def _init():
        env = gym.make(ENV_ID)
        env.reset(seed=seed + rank)
        return env
    return _init

if __name__ == "__main__":
    print("CUDA available:", torch.cuda.is_available())

    set_random_seed(SEED)

    # Create N parallel envs (multiprocess)
    vec_env = SubprocVecEnv([make_env(i, SEED) for i in range(N_ENVS)])
    vec_env = VecMonitor(vec_env)  # logs ep_rew_mean, ep_len_mean

    model = SAC(
        "MlpPolicy",
        vec_env,
        device="cuda",          # 🔥 networks on GPU
        verbose=1,
        learning_rate=3e-4,
        batch_size=1024,        # scale with N_ENVS (try 256–2048)
        train_freq=(1, "step"), # collect 1 step per env then train
        gradient_steps=1,       # keep 1 to avoid over-training per step
        gamma=0.99,
        tau=0.005,
        buffer_size=1_000_000,
        learning_starts=10_000,
    )

    model.learn(total_timesteps=300_000)
    model.save("reacher_sac_vec_gpu")

    vec_env.close()

CUDA available: True
Using cuda device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -40.8    |
| time/              |          |
|    episodes        | 4        |
|    fps             | 2411     |
|    time_elapsed    | 0        |
|    total_timesteps | 400      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -40.8    |
| time/              |          |
|    episodes        | 8        |
|    fps             | 2378     |
|    time_elapsed    | 0        |
|    total_timesteps | 400      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -42      |
| time/              |          |
|    episodes        | 12       |
|    fps             | 2513     |
|    time_elapsed    | 0        |
|    tota

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Streaming output truncated to the last 5000 lines.
|    episodes        | 4520     |
|    fps             | 487      |
|    time_elapsed    | 463      |
|    total_timesteps | 226000   |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -5.48    |
| time/              |          |
|    episodes        | 4524     |
|    fps             | 487      |
|    time_elapsed    | 464      |
|    total_timesteps | 226400   |
| train/             |          |
|    actor_loss      | 9.34     |
|    critic_loss     | 0.0125   |
|    ent_coef        | 0.0208   |
|    ent_coef_loss   | 0.186    |
|    learning_rate   | 0.0003   |
|    n_updates       | 27049    |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -5.48    |
| time/              |          |
|    episodes        | 4528    

In [5]:
import gymnasium as gym
import numpy as np
import pickle
from stable_baselines3 import SAC

model = SAC.load("reacher_sac_vec_gpu")

env = gym.make(ENV_ID)
paths = []
for _ in range(50):
    obs, _ = env.reset()
    observations, actions, rewards, next_obs, terminals = [], [], [], [], []
    for _ in range(200):
        action, _ = model.predict(obs, deterministic=True)
        obs2, rew, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        observations.append(obs); actions.append(action); rewards.append(rew)
        next_obs.append(obs2); terminals.append(float(done))
        obs = obs2
        if done: break
    paths.append({
        "observation": np.array(observations, dtype=np.float32),
        "action": np.array(actions, dtype=np.float32),
        "reward": np.array(rewards, dtype=np.float32),
        "next_observation": np.array(next_obs, dtype=np.float32),
        "terminal": np.array(terminals, dtype=np.float32),
        "image_obs": np.array([], dtype=np.uint8),
    })

with open("reacher_expert.pkl", "wb") as f:
    pickle.dump(paths, f)

env.close()

In [10]:
import os
import gymnasium as gym
import torch
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.utils import set_random_seed

#"Reacher-v4"

ENV_ID = "Humanoid"  #
N_ENVS = 8             # 4–8 is the sweet spot on most machines
SEED = 0

def make_env(rank: int, seed: int = 0):
    def _init():
        env = gym.make(ENV_ID)
        env.reset(seed=seed + rank)
        return env
    return _init

if __name__ == "__main__":
    print("CUDA available:", torch.cuda.is_available())

    set_random_seed(SEED)

    # Create N parallel envs (multiprocess)
    vec_env = SubprocVecEnv([make_env(i, SEED) for i in range(N_ENVS)])
    vec_env = VecMonitor(vec_env)  # logs ep_rew_mean, ep_len_mean

    model = SAC(
        "MlpPolicy",
        vec_env,
        device="cuda",          # 🔥 networks on GPU
        verbose=1,
        learning_rate=3e-4,
        batch_size=1024,        # scale with N_ENVS (try 256–2048)
        train_freq=(1, "step"), # collect 1 step per env then train
        gradient_steps=1,       # keep 1 to avoid over-training per step
        gamma=0.99,
        tau=0.005,
        buffer_size=1_000_000,
        learning_starts=10_000,
    )

    model.learn(total_timesteps=300_000)
    #model.save("reacher_sac_vec_gpu")

    model.save("humanoid_sac_vec_gpu")

    vec_env.close()

Streaming output truncated to the last 5000 lines.
|    n_updates       | 17480    |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 82.2     |
|    ep_rew_mean     | 378      |
| time/              |          |
|    episodes        | 2924     |
|    fps             | 326      |
|    time_elapsed    | 459      |
|    total_timesteps | 150128   |
| train/             |          |
|    actor_loss      | -170     |
|    critic_loss     | 42.9     |
|    ent_coef        | 0.0206   |
|    ent_coef_loss   | 1.42     |
|    learning_rate   | 0.0003   |
|    n_updates       | 17515    |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 80.3     |
|    ep_rew_mean     | 368      |
| time/              |          |
|    episodes        | 2928     |
|    fps             | 326      |
|    time_elapsed    | 459      |
|    total_timesteps | 150336  

In [11]:
import gymnasium as gym
import numpy as np
import pickle
from stable_baselines3 import SAC  # or PPO

ENV_ID = "Humanoid"
MODEL_PATH = "humanoid_sac_vec_gpu"   # change to your saved model
OUT_PKL = "humanoid_expert.pkl"

# How many expert episodes to record
N_EPISODES = 50

# Max steps per episode (Humanoid-v5 has its own TimeLimit; this is a safety cap)
MAX_STEPS = 1000

model = SAC.load(MODEL_PATH)

env = gym.make(ENV_ID)

paths = []
for ep in range(N_EPISODES):
    obs, info = env.reset(seed=ep)
    observations, actions, rewards, next_obs, terminals = [], [], [], [], []

    for t in range(MAX_STEPS):
        action, _ = model.predict(obs, deterministic=True)
        obs2, rew, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        observations.append(obs)
        actions.append(action)
        rewards.append(rew)
        next_obs.append(obs2)
        terminals.append(float(done))

        obs = obs2
        if done:
            break

    paths.append({
        "observation": np.asarray(observations, dtype=np.float32),
        "action": np.asarray(actions, dtype=np.float32),
        "reward": np.asarray(rewards, dtype=np.float32),
        "next_observation": np.asarray(next_obs, dtype=np.float32),
        "terminal": np.asarray(terminals, dtype=np.float32),
        "image_obs": np.asarray([], dtype=np.uint8),  # keep for compatibility
        "env_id": ENV_ID,
    })

    print(f"episode {ep}: steps={len(rewards)} return={float(np.sum(rewards)):.2f}")

with open(OUT_PKL, "wb") as f:
    pickle.dump(paths, f)

env.close()
print(f"Saved {len(paths)} episodes to {OUT_PKL}")

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:520: UserWarning: WARN: Using the latest versioned environment `Humanoid-v5` instead of the unversioned environment `Humanoid`.
  logger.warn(


episode 0: steps=112 return=520.35
episode 1: steps=206 return=986.90
episode 2: steps=145 return=712.12
episode 3: steps=190 return=949.68
episode 4: steps=205 return=1091.14
episode 5: steps=248 return=1190.03
episode 6: steps=165 return=875.37
episode 7: steps=129 return=595.48
episode 8: steps=186 return=850.25
episode 9: steps=168 return=786.00
episode 10: steps=134 return=643.54
episode 11: steps=92 return=405.12
episode 12: steps=213 return=961.10
episode 13: steps=131 return=580.86
episode 14: steps=130 return=652.10
episode 15: steps=199 return=1003.28
episode 16: steps=150 return=704.75
episode 17: steps=194 return=991.00
episode 18: steps=123 return=633.15
episode 19: steps=140 return=716.17
episode 20: steps=197 return=884.71
episode 21: steps=126 return=549.55
episode 22: steps=144 return=699.31
episode 23: steps=181 return=804.51
episode 24: steps=176 return=860.33
episode 25: steps=179 return=886.38
episode 26: steps=160 return=779.22
episode 27: steps=134 return=686.55


Download to macbook. No GUI. Not worth debugging.

In [3]:
import gymnasium as gym
import numpy as np
import pickle

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

from imitation.data import rollout
from imitation.policies.sb3 import SB3Policy
from imitation.data.wrappers import RolloutInfoWrapper


ENV_ID = "Reacher-v5"

def make_env():
    env = gym.make(ENV_ID)
    env = Monitor(env)
    env = RolloutInfoWrapper(env)  # puts episode info into rollout infos
    return env


def train_ppo_expert(venv, total_timesteps=1_000_000, seed=0):
    model = PPO(
        "MlpPolicy",
        venv,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=seed,
        verbose=1,
    )
    model.learn(total_timesteps=total_timesteps)
    return model


def collect_rollouts_to_pkl(sb3_model, venv, out_path, n_episodes=50, seed=0):
    im_policy = SB3Policy(sb3_model)
    rng = np.random.default_rng(seed)

    rollouts = rollout.rollout(
        im_policy,
        venv,
        rollout.make_sample_until(min_episodes=n_episodes),
        rng=rng,
    )

    with open(out_path, "wb") as f:
        pickle.dump(rollouts, f)

    print(f"Saved {len(rollouts)} trajectories to {out_path}")
    return rollouts


# -------------------------
# A) Expert WITHOUT VecNormalize
# -------------------------
venv_no_norm = DummyVecEnv([make_env])

ppo_no_norm = train_ppo_expert(venv_no_norm, total_timesteps=1_000_000, seed=0)
ppo_no_norm.save("ppo_reacher_v5_no_vecnorm")

rollouts_no_norm = collect_rollouts_to_pkl(
    ppo_no_norm,
    venv_no_norm,
    out_path="reacher_v5_expert_no_vecnorm.pkl",
    n_episodes=50,
    seed=0,
)

# -------------------------
# B) Expert WITH VecNormalize (recommended)
# -------------------------
venv_raw = DummyVecEnv([make_env])
venv_vec = VecNormalize(venv_raw, norm_obs=True, norm_reward=True, clip_obs=10.0)

ppo_vec = train_ppo_expert(venv_vec, total_timesteps=1_000_000, seed=0)
ppo_vec.save("ppo_reacher_v5_vecnorm")
venv_vec.save("reacher_v5_vecnorm_stats.pkl")

# Freeze stats for rollout collection
venv_vec.training = False
venv_vec.norm_reward = False

rollouts_vec = collect_rollouts_to_pkl(
    ppo_vec,
    venv_vec,
    out_path="reacher_v5_expert_vecnorm.pkl",
    n_episodes=50,
    seed=0,
)

print("Example rollout object:", rollouts_vec[0])

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


ModuleNotFoundError: No module named 'imitation.policies.sb3'

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
